# Data Cleaning (Fixed) -- Group 8 Flood Risk Prediction

## What changed vs the original notebook

The terrain cleaning steps (drop `flow_dir`, fill risk NaNs, clean `waw`, log-transform `flow_acc`, z-score `dtm`, add `valid_pixel` mask) are unchanged. They are correct.

The ERA5 weather section has been rewritten. The original collapsed all spatial pixels into a single regional mean/max/sum per day, which destroyed within-region spatial variation in weather. That made spatial z-scores impossible and left every terrain pixel in a region sharing the exact same weather values. The fixed version keeps the full ERA5 spatial grid (414 pixels for Severn, 234 for Northumbria) and saves one row per (day, pixel). Feature engineering then computes per-pixel weather features, and at the join step each terrain pixel gets the features of its nearest ERA5 pixel.

## Outputs

| File | Format | Note |
|---|---|---|
| `terrain_severn_clean.nc` | NetCDF | Unchanged from original |
| `terrain_northumbria_clean.nc` | NetCDF | Unchanged from original |
| `era5_severn_daily_spatial.parquet` | Parquet | NEW: one row per (day, ERA5 pixel) for Severn |
| `era5_northumbria_daily_spatial.parquet` | Parquet | NEW: one row per (day, ERA5 pixel) for Northumbria |

The previous `era5_*_daily.parquet` files are no longer produced. The feature engineering notebook reads the new `_spatial.parquet` files instead.


In [1]:
# ── Imports and paths ──────────────────────────────────────────────────────
import xarray as xr
import numpy as np
import pandas as pd
from pathlib import Path

DATA_DIR = Path(r"C:\Users\jackp\Downloads\11_Code_Snippets\Data")
OUT_DIR  = DATA_DIR / "cleaned"
OUT_DIR.mkdir(exist_ok=True)

# Chunk size in pixels per dimension. 2000x2000 = 16 MB per variable per
# chunk in float32, so peak RAM stays manageable on most laptops.
CHUNKS = {"x": 2000, "y": 2000}

RISK_COLS = ["risk_0_2m", "risk_0_3m", "risk_0_6m", "risk_0_9m", "risk_1_2m"]


In [2]:
# ── 1. Lazy load terrain (nothing is read into RAM yet) ────────────────────
terrain_severn = xr.open_dataset(
    DATA_DIR / "flood_risk_terrain_severn.nc", chunks=CHUNKS
)
terrain_northumbria = xr.open_dataset(
    DATA_DIR / "flood_risk_terrain_northumbria.nc", chunks=CHUNKS
)

print("=== Terrain Severn (lazy) ===")
print(terrain_severn)
print(f"\nLogical size: {terrain_severn.nbytes / 1e9:.2f} GB")


=== Terrain Severn (lazy) ===
<xarray.Dataset> Size: 5GB
Dimensions:      (y: 10249, x: 8192)
Coordinates:
  * y            (y) float64 82kB 3.186e+06 3.186e+06 ... 3.391e+06 3.391e+06
  * x            (x) float64 66kB 3.413e+06 3.413e+06 ... 3.577e+06 3.577e+06
    spatial_ref  int64 8B ...
Data variables:
    dtm          (y, x) float32 336MB dask.array<chunksize=(2000, 2000), meta=np.ndarray>
    flow_dir     (y, x) float64 672MB dask.array<chunksize=(2000, 2000), meta=np.ndarray>
    flow_acc     (y, x) float64 672MB dask.array<chunksize=(2000, 2000), meta=np.ndarray>
    imd          (y, x) float32 336MB dask.array<chunksize=(2000, 2000), meta=np.ndarray>
    waw          (y, x) float32 336MB dask.array<chunksize=(2000, 2000), meta=np.ndarray>
    rciw         (y, x) float32 336MB dask.array<chunksize=(2000, 2000), meta=np.ndarray>
    clc_type     (y, x) float32 336MB dask.array<chunksize=(2000, 2000), meta=np.ndarray>
    risk_0_2m    (y, x) float32 336MB dask.array<chunksize=(2

In [3]:
# ── 2. Cleaning function (all operations stay lazy) ────────────────────────
def clean_terrain(ds, region_name):
    """
    Apply all cleaning steps to a terrain dataset. Operations are deferred
    until .to_netcdf() or .compute() is called, so memory usage stays low.
    """
    print(f"\n--- Preparing cleaning operations for {region_name} ---")
    ds = ds.copy()

    # 2a. Drop flow_dir: briefing says it is just a stepping stone to flow_acc
    if "flow_dir" in ds.data_vars:
        ds = ds.drop_vars("flow_dir")
        print("  Dropped flow_dir")

    # 2b. NaN in risk variables means "no risk". Encode as 0 (5th category).
    #     int8 instead of float32 saves 75 percent of memory per risk column.
    for col in RISK_COLS:
        if col in ds.data_vars:
            ds[col] = ds[col].fillna(0).astype("int8")
    print("  Filled NaN with 0 in risk columns and cast to int8")

    # 2c. Clean waw codes:
    #       253 (sea water)      -> recode to 5 (keep as its own category)
    #       254 (unclassifiable) -> NaN
    #       255 (outside area)   -> NaN
    if "waw" in ds.data_vars:
        waw = ds["waw"]
        waw = waw.where(waw != 253, 5)
        waw = waw.where(waw != 254)
        waw = waw.where(waw != 255)
        ds["waw"] = waw
        print("  Cleaned waw sentinel values (253, 254, 255)")

    # 2d. Log-transform flow_acc (heavy right skew on this variable)
    if "flow_acc" in ds.data_vars:
        ds["log_flow_acc"] = np.log1p(ds["flow_acc"]).astype("float32")
        print("  Added log_flow_acc")

    # 2e. Z-score normalize dtm WITHIN this region. Elevation ranges differ
    #     between Severn and Northumbria, so raw dtm would not transfer well
    #     from training to testing. Normalizing per region preserves only
    #     the relative terrain shape, which is what we want the model to learn.
    if "dtm" in ds.data_vars:
        dtm_mean = float(ds["dtm"].mean().compute())
        dtm_std  = float(ds["dtm"].std().compute())
        ds["dtm_zscore"] = ((ds["dtm"] - dtm_mean) / dtm_std).astype("float32")
        print(f"  dtm mean={dtm_mean:.2f}, std={dtm_std:.2f} -> added dtm_zscore")

    # 2f. Validity mask: pixels usable for modeling
    if "dtm" in ds.data_vars and "waw" in ds.data_vars:
        ds["valid_pixel"] = (ds["dtm"].notnull() & ds["waw"].notnull())
        print("  Added valid_pixel mask")

    return ds


def save_cleaned(ds, out_path):
    """Write cleaned dataset chunk by chunk to keep memory low."""
    print(f"  Writing {out_path.name} ...")
    ds.to_netcdf(out_path)
    print(f"  Done.")


In [4]:
# ── 3. Apply terrain cleaning and write to disk ────────────────────────────
ds_severn_clean = clean_terrain(terrain_severn, "Severn")
save_cleaned(ds_severn_clean, OUT_DIR / "terrain_severn_clean.nc")

ds_northumbria_clean = clean_terrain(terrain_northumbria, "Northumbria")
save_cleaned(ds_northumbria_clean, OUT_DIR / "terrain_northumbria_clean.nc")



--- Preparing cleaning operations for Severn ---
  Dropped flow_dir
  Filled NaN with 0 in risk columns and cast to int8
  Cleaned waw sentinel values (253, 254, 255)
  Added log_flow_acc
  dtm mean=1084.44, std=725.49 -> added dtm_zscore
  Added valid_pixel mask
  Writing terrain_severn_clean.nc ...
  Done.

--- Preparing cleaning operations for Northumbria ---
  Dropped flow_dir
  Filled NaN with 0 in risk columns and cast to int8
  Cleaned waw sentinel values (253, 254, 255)
  Added log_flow_acc
  dtm mean=1969.28, std=1532.03 -> added dtm_zscore
  Added valid_pixel mask
  Writing terrain_northumbria_clean.nc ...
  Done.


In [5]:
# ── 4. ERA5 weather: KEEP THE SPATIAL GRID (FIXED VERSION) ─────────────────
#
# Original (broken) version collapsed across all spatial pixels per day,
# producing a single regional time series. That permanently destroyed any
# within-region spatial variation in weather, which made spatial z-scores
# impossible and left every terrain pixel in a region sharing identical
# weather feature values.
#
# Fixed version saves one row per (day, ERA5 pixel). This keeps the spatial
# structure so that:
#   1. Per-pixel weather features can be computed in the next notebook
#   2. Spatial z-scores per slide 24 of the project brief become possible
#   3. Each terrain pixel can be matched to its nearest ERA5 pixel later

era5_severn      = xr.open_dataset(DATA_DIR / "era5_land_severn.nc")
era5_northumbria = xr.open_dataset(DATA_DIR / "era5_land_northumbria.nc")


def save_era5_spatial(ds, region_name, out_path):
    """
    Flatten ERA5 from (time, y, x) into a long dataframe with one row per
    (day, pixel). Renames spatial coords to proj_y / proj_x to match the
    naming used in the terrain parquet output (cleaner downstream joins).
    """
    print(f"\n--- ERA5 spatial flatten for {region_name} ---")
    print(f"  Raw dims: {dict(ds.sizes)}")
    print(f"  Variables: {list(ds.data_vars)}")

    # Identify spatial dim names generically (anything that is not the time dim)
    time_dim = "valid_time"
    spatial_dims = [d for d in ds.dims if d != time_dim]
    if len(spatial_dims) != 2:
        raise ValueError(
            f"Expected 2 spatial dims for ERA5, got: {spatial_dims}"
        )
    y_dim, x_dim = spatial_dims  # order does not matter for the flatten

    # Stack the spatial dims into a single flat "pixel" index
    ds_stacked = ds.stack(pixel=(y_dim, x_dim))
    print(f"  Stacked shape: ({ds_stacked.sizes[time_dim]} days, "
          f"{ds_stacked.sizes['pixel']} pixels)")

    # Convert to a tidy dataframe: each row = one (day, pixel)
    df = ds_stacked.to_dataframe().reset_index()

    # Rename spatial coord columns to a consistent name
    df = df.rename(columns={y_dim: "proj_y", x_dim: "proj_x"})

    # Drop the multiindex helper column if it leaked through
    if "pixel" in df.columns:
        df = df.drop(columns=["pixel"])

    # Add region label and tidy up dtypes for parquet
    df["region"] = region_name
    for col in df.columns:
        if df[col].dtype == "float64":
            df[col] = df[col].astype("float32")

    print(f"  Final shape: {df.shape}")
    print(f"  Columns: {list(df.columns)}")
    print(f"  Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

    df.to_parquet(out_path, index=False)
    print(f"  Saved {out_path}")
    return df


df_era5_sev = save_era5_spatial(
    era5_severn, "severn",
    OUT_DIR / "era5_severn_daily_spatial.parquet"
)
df_era5_nor = save_era5_spatial(
    era5_northumbria, "northumbria",
    OUT_DIR / "era5_northumbria_daily_spatial.parquet"
)

print(f"\nERA5 spatial saved. Severn: {df_era5_sev.shape}, "
      f"Northumbria: {df_era5_nor.shape}")
print(f"Severn unique pixels: {df_era5_sev[['proj_y','proj_x']].drop_duplicates().shape[0]}")
print(f"Northumbria unique pixels: {df_era5_nor[['proj_y','proj_x']].drop_duplicates().shape[0]}")



--- ERA5 spatial flatten for severn ---
  Raw dims: {'valid_time': 3653, 'y': 23, 'x': 18}
  Variables: ['spatial_ref', 'u10_mean', 'v10_mean', 'd2m_mean', 't2m_mean', 'sp_mean', 'swvl1_mean', 'u10_max', 'v10_max', 'd2m_max', 't2m_max', 'sp_max', 'swvl1_max', 'sro', 'tp']
  Stacked shape: (3653 days, 414 pixels)
  Final shape: (1512342, 20)
  Columns: ['valid_time', 'proj_y', 'proj_x', 'spatial_ref', 'u10_mean', 'v10_mean', 'd2m_mean', 't2m_mean', 'sp_mean', 'swvl1_mean', 'u10_max', 'v10_max', 'd2m_max', 't2m_max', 'sp_max', 'swvl1_max', 'sro', 'tp', 'number', 'region']
  Memory: 216.3 MB
  Saved C:\Users\jackp\Downloads\11_Code_Snippets\Data\cleaned\era5_severn_daily_spatial.parquet

--- ERA5 spatial flatten for northumbria ---
  Raw dims: {'valid_time': 3653, 'y': 18, 'x': 13}
  Variables: ['spatial_ref', 'u10_mean', 'v10_mean', 'd2m_mean', 't2m_mean', 'sp_mean', 'swvl1_mean', 'u10_max', 'v10_max', 'd2m_max', 't2m_max', 'sp_max', 'swvl1_max', 'sro', 'tp']
  Stacked shape: (3653 days

In [6]:
# ── 5. Memory-safe summary stats for the cleaned terrain ───────────────────
def summarize_terrain_lazy(path, region_name):
    """Print per-variable summary stats by computing one variable at a time."""
    print(f"\n=== {region_name} cleaned terrain summary ===")
    with xr.open_dataset(path, chunks=CHUNKS) as ds:
        print(f"Dimensions: { {d: ds.sizes[d] for d in ds.dims} }")
        rows = []
        for var in ds.data_vars:
            da = ds[var]
            rows.append({
                "variable": var,
                "dtype":    str(da.dtype),
                "mean":     float(da.mean().compute()) if da.dtype.kind in "fi" else np.nan,
                "min":      float(da.min().compute())  if da.dtype.kind in "fi" else np.nan,
                "max":      float(da.max().compute())  if da.dtype.kind in "fi" else np.nan,
                "n_null":   int(da.isnull().sum().compute()),
            })
        print(pd.DataFrame(rows).set_index("variable").to_string())

summarize_terrain_lazy(OUT_DIR / "terrain_severn_clean.nc",      "Severn")
summarize_terrain_lazy(OUT_DIR / "terrain_northumbria_clean.nc", "Northumbria")

print("\nAll cleaning complete. Cleaned files are in:", OUT_DIR)



=== Severn cleaned terrain summary ===
Dimensions: {'y': 10249, 'x': 8192}
                dtype          mean         min           max    n_null
variable                                                               
dtm           float32  1.084439e+03  -48.000000  7.001000e+03  48681047
flow_acc      float64  1.168822e+05   20.000000  4.648452e+08  48681047
imd           float32  4.786051e+00    0.000000  1.000000e+02  48679322
waw           float32  3.666992e-01    0.000000  5.000000e+00  48679395
rciw          float32  2.313030e+00    2.000000  3.000000e+00  83820403
clc_type      float32  2.218870e+02  111.000000  5.230000e+02  48677882
risk_0_2m        int8  1.050299e-01    0.000000  4.000000e+00         0
risk_0_3m        int8  9.865070e-02    0.000000  4.000000e+00         0
risk_0_6m        int8  8.522244e-02    0.000000  4.000000e+00         0
risk_0_9m        int8  7.595122e-02    0.000000  4.000000e+00         0
risk_1_2m        int8  6.604951e-02    0.000000  4.000000e+0